In [0]:
%sql
-- ================================================================
-- BFSI Lakehouse — Metadata Tables
-- ================================================================

DROP TABLE IF EXISTS bfsi_lakehouse.metadata.schema_drift_log;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.etl_process_log;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.etl_process_master;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.input_column_config;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.table_process_config;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.table_config;


-- ----------------------------------------------------------------
-- 1. table_config
-- Anchor table. One row per source table the pipeline knows about.
-- Holds source identity + Bronze landing target + drift policy.
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.table_config (
    table_id                BIGINT GENERATED ALWAYS AS IDENTITY
                            COMMENT 'Surrogate PK. Auto-generated. Referenced by all child config/log tables.',
    source_table_name       STRING NOT NULL
                            COMMENT 'Logical source table name, e.g. t_Client, t_Loan. Matches folder name in source_synthetic_data/.',
    source_system           STRING NOT NULL
                            COMMENT 'Originating system identifier, e.g. LMS_CORE, CRM. Useful when same table name exists across systems.',
    source_format           STRING NOT NULL
                            COMMENT 'File format of source: PARQUET, CSV, JSON, DELTA. Drives reader selection in ingestion code.',
    source_path_pattern     STRING NOT NULL
                            COMMENT 'Templated path with placeholders, e.g. /Volumes/.../source_synthetic_data/{table}/dt={run_dt}/. Resolved at runtime.',
    target_catalog          STRING NOT NULL
                            COMMENT 'Unity Catalog catalog name for Bronze landing, e.g. bfsi_lakehouse.',
    target_schema           STRING NOT NULL
                            COMMENT 'Unity Catalog schema for Bronze landing, e.g. bronze.',
    target_table            STRING NOT NULL
                            COMMENT 'Bronze table name (typically same as source_table_name, lowercased).',
    partition_cols          STRING
                            COMMENT 'Comma-separated partition columns for target table, e.g. "dt". Bronze partitions by dt only (no batch).',
    replace_where_template  STRING
                            COMMENT 'Templated replaceWhere predicate for idempotent writes, e.g. "dt = ''{run_dt}''". Drives Bronze re-runnability.',
    drift_policy            STRING NOT NULL
                            COMMENT 'How to handle schema drift: STRICT=fail on any change, EVOLVE=auto-add new columns, QUARANTINE=route drifted rows to DLQ. Drives ingestion behavior.',
    load_priority           INT NOT NULL
                            COMMENT 'Execution ordering hint within a process_type. Lower = earlier. Useful when tables have FK-style dependencies (e.g. Client before AccountCustomer).',
    is_active               BOOLEAN NOT NULL
                            COMMENT 'Soft-disable flag. FALSE = skip this table in all pipeline runs without deleting config history.',
    created_at              TIMESTAMP COMMENT 'Audit: when this config row was first inserted.',
    created_by              STRING COMMENT 'Audit: user/service that created this config row.',
    last_edited_at          TIMESTAMP COMMENT 'Audit: last modification timestamp.',
    last_edited_by          STRING COMMENT 'Audit: last modifier.',
    CONSTRAINT tc_pk PRIMARY KEY (table_id)
) USING DELTA
COMMENT 'Master config: one row per source table. Defines source location, Bronze target, drift policy. Immutable definition — runtime state lives in etl_process_log.'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.table_config
    ADD CONSTRAINT tc_chk_drift
    CHECK (drift_policy IN ('STRICT','EVOLVE','QUARANTINE'));


-- ----------------------------------------------------------------
-- 2. table_process_config
-- Defines WHAT processing applies to each table at each layer.
-- One table_id can have multiple rows: one per layer (BRONZE/SILVER/GOLD).
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.table_process_config (
    process_config_id     BIGINT GENERATED ALWAYS AS IDENTITY
                          COMMENT 'Surrogate PK.',
    table_id              BIGINT NOT NULL
                          COMMENT 'FK to table_config.table_id.',
    process_type          STRING NOT NULL
                          COMMENT 'Layer this config applies to: BRONZE, SILVER, or GOLD. Same table_id can have multiple rows (one per layer).',
    load_type             STRING NOT NULL
                          COMMENT 'Load semantics: FULL=full refresh (truncate+load), INCREMENTAL=append/merge new data only. Drives reader watermark logic.',
    depends_on_table_ids  STRING
                          COMMENT 'Comma-separated table_ids this layer depends on, e.g. Silver t_Loan depends on Silver t_AccountCustomer. Drives DAG ordering.',
    transform_module      STRING
                          COMMENT 'Python module path containing the transformation function for this layer, e.g. silver.transforms.t_loan. NULL for Bronze (as-is ingestion).',
    is_active             BOOLEAN NOT NULL
                          COMMENT 'Soft-disable flag for this specific layer of this table.',
    created_at            TIMESTAMP COMMENT 'Audit.',
    created_by            STRING    COMMENT 'Audit.',
    last_edited_at        TIMESTAMP COMMENT 'Audit.',
    last_edited_by        STRING    COMMENT 'Audit.',
    CONSTRAINT tpc_pk       PRIMARY KEY (process_config_id),
    CONSTRAINT tpc_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
COMMENT 'Per-layer processing rules. Maps a source table to its Bronze/Silver/Gold processing config, dependencies, and transform module.'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.table_process_config
    ADD CONSTRAINT tpc_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.table_process_config
    ADD CONSTRAINT tpc_chk_load CHECK (load_type IN ('FULL','INCREMENTAL'));


-- ----------------------------------------------------------------
-- 3. input_column_config
-- Column-level metadata. Scoped to Silver/Gold (Bronze is as-is).
-- Drives schema definition, PII masking, type casting, derived columns.
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.input_column_config (
    column_id           BIGINT GENERATED ALWAYS AS IDENTITY
                        COMMENT 'Surrogate PK.',
    table_id            BIGINT NOT NULL
                        COMMENT 'FK to table_config.table_id.',
    process_type        STRING NOT NULL
                        COMMENT 'Layer this column definition applies to. Bronze ingests as-is, so rows here are typically SILVER/GOLD.',
    source_column_name  STRING NOT NULL
                        COMMENT 'Source column name from source table',
    target_column_name  STRING
                        COMMENT 'Target column name in the output table(Rename). NULL if Same',
    data_type           STRING NOT NULL
                        COMMENT 'Target Spark data type, e.g. STRING, BIGINT, DECIMAL(15,2), TIMESTAMP. Financial cols MUST use DECIMAL(15,2) — never DOUBLE.',
    is_nullable         BOOLEAN
                        COMMENT 'Schema enforcement flag. FALSE = reject row if null at write time.',
    is_pii              BOOLEAN
                        COMMENT 'PII flag. Drives masking/tokenization logic and access control via Unity Catalog row/column filters.',
    column_purpose      STRING
                        COMMENT 'Semantic role: KEY=business/join key, ATTRIBUTE=descriptive field, AUDIT=created_at/updated_by etc., DERIVED=computed via transform_expr.',
    is_active           BOOLEAN NOT NULL
                        COMMENT 'Soft-disable flag. FALSE = exclude column from output schema.',
    created_at          TIMESTAMP COMMENT 'Audit.',
    created_by          STRING    COMMENT 'Audit.',
    last_edited_at      TIMESTAMP COMMENT 'Audit.',
    last_edited_by      STRING    COMMENT 'Audit.',
    CONSTRAINT icc_pk       PRIMARY KEY (column_id),
    CONSTRAINT icc_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
COMMENT 'Column-level config driving Silver/Gold schema definition, type casting, PII flags, and derived column logic. Bronze ingests as-is and typically has no rows here.'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.input_column_config
    ADD CONSTRAINT icc_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.input_column_config
    ADD CONSTRAINT icc_chk_purpose CHECK (column_purpose IN ('KEY','ATTRIBUTE','AUDIT','DERIVED'));


-- ----------------------------------------------------------------
-- 4. etl_process_master
-- One row per pipeline RUN (across all tables).
-- Tracks the overall job. Child rows in etl_process_log track per-table.
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.etl_process_master (
    run_id            STRING NOT NULL
                      COMMENT 'PK. UUID generated at job start. Also stamped on every Bronze row as _ingestion_run_id for lineage.',
    pipeline_name     STRING NOT NULL
                      COMMENT 'Logical pipeline name, e.g. "bronze_eod_ingest", "silver_daily_transform".',
    process_type      STRING NOT NULL
                      COMMENT 'Layer this run targets: BRONZE, SILVER, or GOLD.',
    run_dt            STRING NOT NULL
                      COMMENT 'Business date being processed (yyyy-MM-dd). Distinct from started_at (wall clock). Backfill runs have run_dt in the past.',
    trigger_type      STRING NOT NULL
                      COMMENT 'SCHEDULED=Databricks Workflow cron, MANUAL=user-invoked, BACKFILL=historical replay.',
    triggered_by      STRING NOT NULL
                      COMMENT 'User or service principal that initiated the run.',
    status            STRING NOT NULL
                      COMMENT 'STARTED=in flight, SUCCESS=all tables succeeded, FAILED=fatal abort, PARTIAL=some tables succeeded, some failed.',
    total_tables      INT    COMMENT 'Count of tables this run intended to process.',
    success_count     INT    COMMENT 'Tables completed successfully.',
    failed_count      INT    COMMENT 'Tables that errored.',
    skipped_count     INT    COMMENT 'Tables skipped (e.g. is_active=false, no new data).',
    started_at        TIMESTAMP NOT NULL
                      COMMENT 'Wall-clock run start.',
    finished_at       TIMESTAMP
                      COMMENT 'Wall-clock run end. NULL while STARTED.',
    duration_seconds  DOUBLE
                      COMMENT 'Total run duration. Populated at finalization.',
    error_message     STRING
                      COMMENT 'Top-level error if status=FAILED. Per-table errors live in etl_process_log.',
    created_by        STRING    COMMENT 'Source of insert, e.g. databricks_workflow, manual.',
    created_at        TIMESTAMP COMMENT 'Row insert time.',
    updated_at        TIMESTAMP COMMENT 'Last update time (status transitions).',
    CONSTRAINT epm_pk PRIMARY KEY (run_id)
) USING DELTA
COMMENT 'Pipeline run header. One row per job execution. Parent of etl_process_log (per-table detail) and schema_drift_log.'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.etl_process_master
    ADD CONSTRAINT epm_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_master
    ADD CONSTRAINT epm_chk_trigger CHECK (trigger_type IN ('SCHEDULED','MANUAL','BACKFILL'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_master
    ADD CONSTRAINT epm_chk_status CHECK (status IN ('STARTED','SUCCESS','FAILED','PARTIAL'));


-- ----------------------------------------------------------------
-- 5. etl_process_log
-- One row per (run_id, table_id). Per-table execution detail.
-- THIS is the watermark source — last SUCCESS row per table gives next run_dt.
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.etl_process_log (
    log_id                STRING NOT NULL
                          COMMENT 'PK. UUID per (run_id, table_id).',
    run_id                STRING NOT NULL
                          COMMENT 'FK to etl_process_master.run_id.',
    table_id              BIGINT NOT NULL
                          COMMENT 'FK to table_config.table_id.',
    table_name            STRING NOT NULL
                          COMMENT 'Denormalized for readability in queries — avoids join to table_config for ad-hoc inspection.',
    process_type          STRING NOT NULL
                          COMMENT 'Layer: BRONZE, SILVER, or GOLD. Matches CHECK constraint values (BRONZE/SILVER/GOLD), not the older BRONZE_INGEST style.',
    load_type             STRING NOT NULL
                          COMMENT 'FULL or INCREMENTAL. Same semantics as table_process_config.load_type.',
    run_dt                STRING NOT NULL
                          COMMENT 'Business date processed. Watermark queries filter on MAX(run_dt) WHERE status=SUCCESS.',
    status                STRING NOT NULL
                          COMMENT 'STARTED=in flight, SUCCESS=committed, FAILED=errored, SKIPPED=intentionally bypassed (no new data, inactive config).',
    rows_read             BIGINT COMMENT 'Source row count read this run. Useful for DQ drift detection.',
    rows_written          BIGINT COMMENT 'Rows written to target. May differ from rows_read due to dedup/filter.',
    delta_version_before  BIGINT COMMENT 'Target Delta table version before this write. Enables time-travel rollback to pre-run state.',
    delta_version_after   BIGINT COMMENT 'Target Delta table version after commit. Pair with _before for exact lineage.',
    started_at            TIMESTAMP NOT NULL COMMENT 'Per-table start.',
    finished_at           TIMESTAMP COMMENT 'Per-table end. NULL while STARTED.',
    heartbeat_at          TIMESTAMP COMMENT 'Updated periodically during long runs. Orphan detection: if heartbeat is stale and status=STARTED, run likely crashed.',
    -- NOTE: removed generated duration_seconds column.
    -- CAST(timestamp AS LONG) is not accepted as deterministic by Delta generated columns,
    -- and finished_at is populated later via UPDATE — generated columns only evaluate on INSERT.
    -- Compute duration in the UPDATE that sets finished_at, or as a view.
    duration_seconds      BIGINT COMMENT 'Wall-clock duration in seconds. Computed and written at finalization (UPDATE step).',
    error_message         STRING COMMENT 'Short error description if FAILED.',
    error_stacktrace      STRING COMMENT 'Full stacktrace for debugging.',
    created_by            STRING    COMMENT 'Source of insert.',
    created_at            TIMESTAMP COMMENT 'Row insert time.',
    updated_at            TIMESTAMP COMMENT 'Last update time.',
    CONSTRAINT epl_pk       PRIMARY KEY (log_id),
    CONSTRAINT epl_fk_run   FOREIGN KEY (run_id)
                            REFERENCES bfsi_lakehouse.metadata.etl_process_master(run_id),
    CONSTRAINT epl_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
COMMENT 'Per-table execution log. Authoritative source for watermark (last SUCCESS run_dt per table). Drives idempotency and backfill detection.'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.etl_process_log
    ADD CONSTRAINT epl_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_log
    ADD CONSTRAINT epl_chk_load CHECK (load_type IN ('FULL','INCREMENTAL'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_log
    ADD CONSTRAINT epl_chk_status CHECK (status IN ('STARTED','SUCCESS','FAILED','SKIPPED'));


-- ----------------------------------------------------------------
-- 6. schema_drift_log
-- One row per drift event detected during ingestion.
-- Driven by table_config.drift_policy.
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.schema_drift_log (
    drift_log_id      STRING NOT NULL
                      COMMENT 'PK. UUID per drift event.',
    run_id            STRING NOT NULL
                      COMMENT 'FK to etl_process_master.run_id — drift events tied to the run that detected them.',
    table_id          BIGINT NOT NULL
                      COMMENT 'FK to table_config.table_id.',
    column_name       STRING NOT NULL
                      COMMENT 'The column that drifted.',
    column_data_type  STRING
                      COMMENT 'Detected data type at source. For TYPE_CHANGED, the NEW type; old type is implied by target schema.',
    event_type        STRING NOT NULL
                      COMMENT 'NEW=column exists in source but not target, MISSING=column in target but not in source, TYPE_CHANGED=type mismatch.',
    action_taken      STRING NOT NULL
                      COMMENT 'Action driven by drift_policy: AUTO_EVOLVED (policy=EVOLVE, target schema updated), REJECTED (policy=STRICT, run failed), QUARANTINED (policy=QUARANTINE, rows routed to DLQ).',
    detected_at       TIMESTAMP NOT NULL
                      COMMENT 'When drift was first observed in this run.',
    created_by        STRING    COMMENT 'Source of insert.',
    created_at        TIMESTAMP COMMENT 'Row insert time.',
    updated_at        TIMESTAMP COMMENT 'Last update time.',
    CONSTRAINT sdl_pk       PRIMARY KEY (drift_log_id),
    CONSTRAINT sdl_fk_run   FOREIGN KEY (run_id)
                            REFERENCES bfsi_lakehouse.metadata.etl_process_master(run_id),
    CONSTRAINT sdl_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
COMMENT 'Schema drift event log. Action mapping: drift_policy=STRICT → REJECTED, EVOLVE → AUTO_EVOLVED, QUARANTINE → QUARANTINED.'
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.schema_drift_log
    ADD CONSTRAINT sdl_chk_event CHECK (event_type IN ('NEW','MISSING','TYPE_CHANGED'));

ALTER TABLE bfsi_lakehouse.metadata.schema_drift_log
    ADD CONSTRAINT sdl_chk_action CHECK (action_taken IN ('AUTO_EVOLVED','REJECTED','QUARANTINED'));

In [0]:
%sql

-- ================================================================
-- BFSI Lakehouse — Metadata Tables
-- ================================================================

DROP TABLE IF EXISTS bfsi_lakehouse.metadata.schema_drift_log;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.etl_process_log;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.etl_process_master;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.input_column_config;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.table_process_config;
DROP TABLE IF EXISTS bfsi_lakehouse.metadata.table_config;


-- ----------------------------------------------------------------
-- 1. table_config
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.table_config (
    table_id              BIGINT GENERATED ALWAYS AS IDENTITY,
    source_table_name     STRING NOT NULL,
    source_system         STRING NOT NULL,
    source_format         STRING NOT NULL,
    source_path_pattern   STRING NOT NULL,
    target_catalog        STRING NOT NULL,
    target_schema         STRING NOT NULL,
    target_table          STRING NOT NULL,
    partition_cols        STRING,
    replace_where_template STRING,
    drift_policy          STRING NOT NULL,
    load_priority         INT NOT NULL,
    is_active             BOOLEAN NOT NULL,
    created_at            TIMESTAMP,
    created_by            STRING,
    last_edited_at        TIMESTAMP,
    last_edited_by        STRING,
    CONSTRAINT tc_pk PRIMARY KEY (table_id)
) USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.table_config
    ADD CONSTRAINT tc_chk_drift
    CHECK (drift_policy IN ('STRICT','EVOLVE','QUARANTINE'));


-- ----------------------------------------------------------------
-- 2. table_process_config
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.table_process_config (
    process_config_id     BIGINT GENERATED ALWAYS AS IDENTITY,
    table_id              BIGINT NOT NULL,
    process_type          STRING NOT NULL,
    load_type             STRING NOT NULL,
    depends_on_table_ids  STRING,
    transform_module      STRING,
    is_active             BOOLEAN NOT NULL,
    created_at            TIMESTAMP,
    created_by            STRING,
    last_edited_at        TIMESTAMP,
    last_edited_by        STRING,
    CONSTRAINT tpc_pk       PRIMARY KEY (process_config_id),
    CONSTRAINT tpc_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.table_process_config
    ADD CONSTRAINT tpc_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.table_process_config
    ADD CONSTRAINT tpc_chk_load CHECK (load_type IN ('FULL','INCREMENTAL'));


-- ----------------------------------------------------------------
-- 3. input_column_config
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.input_column_config (
    column_id             BIGINT GENERATED ALWAYS AS IDENTITY,
    table_id              BIGINT NOT NULL,
    process_type          STRING NOT NULL,
    column_name           STRING NOT NULL,
    source_column_name    STRING,
    data_type             STRING NOT NULL,
    is_nullable           BOOLEAN,
    is_pii                BOOLEAN,
    column_purpose        STRING,
    transform_expr        STRING,
    is_active             BOOLEAN NOT NULL,
    created_at            TIMESTAMP,
    created_by            STRING,
    last_edited_at        TIMESTAMP,
    last_edited_by        STRING,
    CONSTRAINT icc_pk       PRIMARY KEY (column_id),
    CONSTRAINT icc_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.input_column_config
    ADD CONSTRAINT icc_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.input_column_config
    ADD CONSTRAINT icc_chk_purpose CHECK (column_purpose IN ('KEY','ATTRIBUTE','AUDIT','DERIVED'));


-- ----------------------------------------------------------------
-- 4. etl_process_master
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.etl_process_master (
    run_id                STRING NOT NULL,
    pipeline_name         STRING NOT NULL,
    process_type          STRING NOT NULL,
    run_dt                STRING NOT NULL,
    trigger_type          STRING NOT NULL,
    triggered_by          STRING NOT NULL,
    status                STRING NOT NULL,
    total_tables          INT,
    success_count         INT,
    failed_count          INT,
    skipped_count         INT,
    started_at            TIMESTAMP NOT NULL,
    finished_at           TIMESTAMP,
    duration_seconds      DOUBLE,
    error_message         STRING,
    created_by            STRING,           -- 'databricks_workflow', 'manual', etc.
    created_at            TIMESTAMP,
    updated_at            TIMESTAMP,
    CONSTRAINT epm_pk PRIMARY KEY (run_id)
) USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.etl_process_master
    ADD CONSTRAINT epm_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_master
    ADD CONSTRAINT epm_chk_trigger CHECK (trigger_type IN ('SCHEDULED','MANUAL','BACKFILL'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_master
    ADD CONSTRAINT epm_chk_status CHECK (status IN ('STARTED','SUCCESS','FAILED','PARTIAL'));


-- ----------------------------------------------------------------
-- 5. etl_process_log
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.etl_process_log (
    log_id                STRING NOT NULL,
    run_id                STRING NOT NULL,
    table_id              BIGINT NOT NULL,
    table_name            STRING NOT NULL,
    process_type          STRING NOT NULL,  -- 'BRONZE_INGEST', 'SILVER_TRANSFORM', 'GOLD_AGGREGATE'
    load_type             STRING NOT NULL,  -- 'EOD', 'INTRADAY', 'BACKFILL'
    run_dt                STRING NOT NULL,  -- business date being processed
    status                STRING NOT NULL,  -- 'RUNNING', 'SUCCESS', 'FAILED', 'SKIPPED'
    rows_read             BIGINT,
    rows_written          BIGINT,
    delta_version_before  BIGINT,
    delta_version_after   BIGINT,
    started_at            TIMESTAMP NOT NULL,
    finished_at           TIMESTAMP,
    heartbeat_at          TIMESTAMP,        -- updated on every log call; orphan detection
    duration_seconds      BIGINT GENERATED ALWAYS AS (CAST(finished_at AS LONG) - CAST(started_at AS LONG)),
    error_message         STRING,
    error_stacktrace      STRING,
    created_by            STRING,           -- 'databricks_workflow', 'manual', etc.
    created_at            TIMESTAMP,
    updated_at            TIMESTAMP,
    CONSTRAINT epl_pk       PRIMARY KEY (log_id),
    CONSTRAINT epl_fk_run   FOREIGN KEY (run_id)
                            REFERENCES bfsi_lakehouse.metadata.etl_process_master(run_id),
    CONSTRAINT epl_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.etl_process_log
    ADD CONSTRAINT epl_chk_process CHECK (process_type IN ('BRONZE','SILVER','GOLD'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_log
    ADD CONSTRAINT epl_chk_load CHECK (load_type IN ('FULL','INCREMENTAL'));

ALTER TABLE bfsi_lakehouse.metadata.etl_process_log
    ADD CONSTRAINT epl_chk_status CHECK (status IN ('STARTED','SUCCESS','FAILED','SKIPPED'));


-- ----------------------------------------------------------------
-- 6. schema_drift_log
-- ----------------------------------------------------------------
CREATE TABLE IF NOT EXISTS bfsi_lakehouse.metadata.schema_drift_log (
    drift_log_id          STRING NOT NULL,
    run_id                STRING NOT NULL,
    table_id              BIGINT NOT NULL,
    column_name           STRING NOT NULL,
    column_data_type      STRING,
    event_type            STRING NOT NULL,
    action_taken          STRING NOT NULL,
    detected_at           TIMESTAMP NOT NULL,
    created_by            STRING,           -- 'databricks_workflow', 'manual', etc.
    created_at            TIMESTAMP,
    updated_at            TIMESTAMP,
    CONSTRAINT sdl_pk       PRIMARY KEY (drift_log_id),
    CONSTRAINT sdl_fk_run   FOREIGN KEY (run_id)
                            REFERENCES bfsi_lakehouse.metadata.etl_process_master(run_id),
    CONSTRAINT sdl_fk_table FOREIGN KEY (table_id)
                            REFERENCES bfsi_lakehouse.metadata.table_config(table_id)
) USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

ALTER TABLE bfsi_lakehouse.metadata.schema_drift_log
    ADD CONSTRAINT sdl_chk_event CHECK (event_type IN ('NEW','MISSING','TYPE_CHANGED'));

ALTER TABLE bfsi_lakehouse.metadata.schema_drift_log
    ADD CONSTRAINT sdl_chk_action CHECK (action_taken IN ('AUTO_EVOLVED','REJECTED','QUARANTINED'));